# HKH Glacier Mass-Change Classification — Unified Chronological Hyperparameter Tuning

This notebook retains **all 697 HKH grid locations** and the complete supplied 1991–2024 dataset.

## Research design

- **Target:** `mass_change_class` (mass gain / mass loss / near balance)
- **Final untouched test period:** hydrological years 2020–2024
- **Development period:** all years before 2020
- **Algorithms:** Logistic Regression, RBF-SVM, Random Forest, XGBoost
- **Imbalance settings:** undersampling `{False, True}` × class weighting `{False, True}`
- **Search:** one unified experiment loop over all **16 algorithm/imbalance combinations**, with algorithm-specific hyperparameters sampled in the same framework
- **Validation:** identical expanding-window chronological folds for every candidate
- **Primary selection metric:** mean validation Macro F1
- **Secondary metrics:** mean validation balanced accuracy, then accuracy
- **Outputs:** consolidated CSV reports, test confusion matrices, ROC curves, train-vs-validation plots, and selected model saved with Joblib

**Important:** The test set is not used for tuning, selection, feature selection, or preprocessing decisions. Every fold's validation data and the final test set retain their natural class distributions. Undersampling is applied only to each training fold (and later to the full development training set for final fitting).

## Important interpretation

The 2020–2024 holdout evaluates **future-year performance at grid locations represented in the study data**. It does not establish transfer to unseen locations. Spatial autocorrelation between HKH grid cells may make ordinary row-level metrics optimistic for geographic transfer; consider a separate spatial-block validation study.

The class label is derived from `glacier_mass_change_mwe` using thresholds of +0.1 and −0.1 m w.e. This measured value is excluded from the predictors to prevent direct target leakage.

## 1. Install dependencies

Run this cell if the required packages are not already installed. XGBoost and imbalanced-learn are required.

In [ ]:
# Uncomment if required:
# %pip install -U pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost joblib

## 2. Imports and reproducibility

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterSampler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, classification_report,
    confusion_matrix, roc_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

OUTPUT_DIR = Path("glacier_classification_outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Imports completed.")

## 3. Configuration and dataset loading

Place `processed_glacier_data.csv` in the same directory as this notebook, or update `DATA_PATH`. The code deliberately does not silently substitute another dataset.

In [ ]:
DATA_PATH = Path("processed_glacier_data.csv")
# Example Windows path:
# DATA_PATH = Path(r"E:\glacier_research\processed_data\processed_glacier_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Place processed_glacier_data.csv beside this notebook or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
display(df.dtypes.rename("dtype").to_frame())

## 4. Validate the dataset and construct the classification target

The supplied file already contains `mass_change_class`. The notebook independently reconstructs it from the continuous mass-change target and checks for disagreement. This prevents silently training against inconsistent labels.

All 697 HKH grid locations are retained; no Nepal-only geographic filter is applied.

In [ ]:
TARGET = "mass_change_class"
MASS_COL = "glacier_mass_change_mwe"
TIME_COL = "hydrological_year"
LAT_COL = "target_latitude"
LON_COL = "target_longitude"

required_cols = {TARGET, MASS_COL, TIME_COL, LAT_COL, LON_COL}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Required columns missing: {sorted(missing)}")

if df.isna().any().any():
    missing_counts = df.isna().sum()
    raise ValueError(
        "Missing values detected. Resolve them explicitly before modelling:\n"
        + missing_counts[missing_counts > 0].to_string()
    )

conditions = [df[MASS_COL] > 0.1, df[MASS_COL] < -0.1]
labels = ["mass gain", "mass loss"]
reconstructed = np.select(conditions, labels, default="near balance")

if not (df[TARGET].astype(str).to_numpy() == reconstructed).all():
    mismatch_count = int((df[TARGET].astype(str).to_numpy() != reconstructed).sum())
    raise ValueError(
        f"{mismatch_count} supplied labels disagree with labels reconstructed "
        "from glacier_mass_change_mwe and ±0.1 m w.e. thresholds."
    )

df[TARGET] = reconstructed
df[TIME_COL] = pd.to_numeric(df[TIME_COL], errors="raise").astype(int)

n_locations = df[[LAT_COL, LON_COL]].drop_duplicates().shape[0]
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique HKH grid locations:", n_locations)
print("Hydrological years:", df[TIME_COL].min(), "to", df[TIME_COL].max())
print("Missing values:", int(df.isna().sum().sum()))
print("\nClass distribution:")
display(df[TARGET].value_counts().rename_axis("class").to_frame("count"))
display((df[TARGET].value_counts(normalize=True) * 100).round(2).rename("percent").to_frame())

if n_locations != 697:
    print(f"WARNING: expected 697 grid locations based on the supplied dataset review; found {n_locations}.")

## 5. Exploratory class-distribution checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df[TARGET].value_counts().reindex(df[TARGET].value_counts().index).plot(kind="bar", ax=axes[0])
axes[0].set_title("Overall HKH target-class distribution")
axes[0].set_xlabel("Mass-change class")
axes[0].set_ylabel("Observations")
axes[0].tick_params(axis="x", rotation=20)

year_class = pd.crosstab(df[TIME_COL], df[TARGET])
year_class.plot(kind="bar", stacked=True, ax=axes[1], width=0.85)
axes[1].set_title("Class distribution by hydrological year")
axes[1].set_xlabel("Hydrological year")
axes[1].set_ylabel("Observations")
axes[1].legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

## 6. Feature selection and leakage controls

Excluded from predictors:
- `mass_change_class`: target label.
- `glacier_mass_change_mwe`: continuous measurement used to construct the target.
- `uncertainty_mwe`: target-measurement uncertainty, not an independent climate predictor.

The remaining numeric fields are used as predictors. These include coordinates, glacier area, hydrological year and ERA5-Land variables. Confirm that every predictor is available at the intended prediction time. In particular, using full-hydrological-year climate aggregates means this is not an early-season forecast unless those aggregates are available then.

`hydrological_year`, latitude and longitude can encode temporal and spatial structure. Their use is retained as in the original study design, but results should not be interpreted as performance on unseen years/locations without additional tests.

In [ ]:
LEAKAGE_COLUMNS = [TARGET, MASS_COL, "uncertainty_mwe"]
feature_cols = [c for c in df.columns if c not in LEAKAGE_COLUMNS]

non_numeric = df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise TypeError(f"Non-numeric predictors require explicit handling: {non_numeric}")

X = df[feature_cols].copy()
y_text = df[TARGET].astype(str).copy()
years = df[TIME_COL].astype(int).copy()

classes = sorted(y_text.unique())
class_to_int = {label: i for i, label in enumerate(classes)}
int_to_class = {i: label for label, i in class_to_int.items()}
y = y_text.map(class_to_int).astype(int)

print("Number of predictors:", len(feature_cols))
print("Predictors:", feature_cols)
print("Class mapping:", class_to_int)

## 7. Untouched chronological test split

All rows from hydrological years 2020–2024 are held out. Model tuning uses only years before 2020. No test metrics are computed until the selected configurations have been determined from validation.

In [ ]:
TEST_START_YEAR = 2020
all_years = sorted(years.unique())
dev_years = [yr for yr in all_years if yr < TEST_START_YEAR]
test_years = [yr for yr in all_years if yr >= TEST_START_YEAR]

if not dev_years or not test_years:
    raise ValueError("Development or test period is empty.")
if min(test_years) != TEST_START_YEAR or max(test_years) != 2024:
    raise ValueError(f"Expected test years 2020–2024; found {min(test_years)}–{max(test_years)}.")

dev_mask = years.isin(dev_years)
test_mask = years.isin(test_years)

X_dev = X.loc[dev_mask].reset_index(drop=True)
y_dev = y.loc[dev_mask].reset_index(drop=True)
year_dev = years.loc[dev_mask].reset_index(drop=True)

X_test = X.loc[test_mask].reset_index(drop=True)
y_test = y.loc[test_mask].reset_index(drop=True)
year_test = years.loc[test_mask].reset_index(drop=True)

print(f"Development: {min(dev_years)}–{max(dev_years)} | n={len(X_dev):,}")
print(f"Untouched test: {min(test_years)}–{max(test_years)} | n={len(X_test):,}")
print("\nDevelopment classes:")
display(y_dev.map(int_to_class).value_counts().rename("count").to_frame())
print("\nTest classes (natural distribution):")
display(y_test.map(int_to_class).value_counts().rename("count").to_frame())

## 8. Create one shared expanding-window fold schedule

Every candidate in all 16 combinations uses these exact fold indices. Each validation block is later in time than its training data. The final short block is allowed when fewer than three years remain.

In [ ]:
def make_expanding_folds(year_series, min_train_years=12, validation_years=3, step_years=3):
    unique_years = sorted(pd.Series(year_series).unique())
    folds = []
    first_validation_position = min_train_years

    while first_validation_position < len(unique_years):
        train_yrs = unique_years[:first_validation_position]
        val_yrs = unique_years[
            first_validation_position:first_validation_position + validation_years
        ]
        if not val_yrs:
            break

        train_idx = np.flatnonzero(pd.Series(year_series).isin(train_yrs).to_numpy())
        val_idx = np.flatnonzero(pd.Series(year_series).isin(val_yrs).to_numpy())

        if max(train_yrs) >= min(val_yrs):
            raise AssertionError("Chronological fold order violated.")

        folds.append((train_idx, val_idx, train_yrs, val_yrs))
        first_validation_position += step_years

    return folds

folds = make_expanding_folds(
    year_dev, min_train_years=12, validation_years=3, step_years=3
)
if len(folds) < 2:
    raise ValueError("Not enough development years for at least two expanding-window folds.")

fold_summary = pd.DataFrame([
    {
        "fold": fold_id,
        "training_period": f"{min(train_yrs)}–{max(train_yrs)}",
        "validation_period": f"{min(val_yrs)}–{max(val_yrs)}",
        "n_train": len(train_idx),
        "n_validation": len(val_idx),
        "validation_classes": ", ".join(
            sorted(y_dev.iloc[val_idx].map(int_to_class).unique())
        )
    }
    for fold_id, (train_idx, val_idx, train_yrs, val_yrs) in enumerate(folds, 1)
])
display(fold_summary)

## 9. Unified search space and imbalance settings

Each algorithm has its own parameter distribution, but the same search loop evaluates every combination of:
- `undersampling`: False / True
- `class_weighting`: False / True

The default is 12 sampled hyperparameter configurations **per algorithm × imbalance combination**: 4 × 2 × 2 × 12 = 192 candidate configurations. Each candidate is evaluated on every shared fold.

**Weighting definition:** when enabled, balanced class weights are calculated from the original, pre-undersampling training fold. If undersampling is also enabled, those original-fold weights are applied to the retained observations. This makes the two toggles genuinely distinct, but the combined strategy can emphasize minority examples strongly; compare it empirically rather than assuming it is superior.

In [ ]:
N_ITER = 4  # 64 candidates total by default; increase (e.g. 8 or 12) for a broader search.
RANDOM_STATE = 42

ALGORITHMS = [
    "Logistic Regression",
    "SVM (RBF)",
    "Random Forest",
    "XGBoost",
]
IMBALANCE_SETTINGS = [
    {"undersampling": under, "class_weighting": weight}
    for under in [False, True]
    for weight in [False, True]
]

PARAM_DISTRIBUTIONS = {
    "Logistic Regression": {
        "C": np.logspace(-3, 2, 12),
        "solver": ["lbfgs"],
    },
    "SVM (RBF)": {
        "C": [0.1, 0.5, 1, 2, 5, 10, 20, 50],
        "gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1.0],
    },
    "Random Forest": {
        "n_estimators": [200, 350, 500, 700],
        "max_depth": [None, 8, 12, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.7],
    },
    "XGBoost": {
        "n_estimators": [150, 250, 350, 500],
        "max_depth": [2, 3, 4, 5],
        "learning_rate": [0.02, 0.05, 0.1],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "min_child_weight": [1, 3, 5],
        "reg_lambda": [1, 3, 5, 10],
    },
}

print("Algorithm/imbalance combinations:", len(ALGORITHMS) * len(IMBALANCE_SETTINGS))
print("Candidate configurations:", len(ALGORITHMS) * len(IMBALANCE_SETTINGS) * N_ITER)
print("Fold fits:", len(ALGORITHMS) * len(IMBALANCE_SETTINGS) * N_ITER * len(folds))

## 10. Model factory and fold-only imbalance handling

Scaling is fitted inside a pipeline for Logistic Regression and SVM only. Tree models do not need feature scaling. Random undersampling is applied only to the training partition. Validation and test rows are never resampled.

In [ ]:
def original_fold_class_weights(y_original):
    present_classes = np.unique(y_original)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=np.asarray(y_original)
    )
    return {int(cls): float(weight) for cls, weight in zip(present_classes, weights)}

def build_estimator(algorithm, params, class_weights=None):
    params = dict(params)

    if algorithm == "Logistic Regression":
        params.pop("solver", None)
        estimator = LogisticRegression(
            max_iter=4000,
            solver="lbfgs",
            class_weight=class_weights,
            random_state=RANDOM_STATE,
            **params
        )
        return Pipeline([("scaler", StandardScaler()), ("model", estimator)])

    if algorithm == "SVM (RBF)":
        estimator = SVC(
            kernel="rbf",
            probability=False,
            decision_function_shape="ovr",
            cache_size=1000,
            class_weight=class_weights,
            random_state=RANDOM_STATE,
            **params
        )
        return Pipeline([("scaler", StandardScaler()), ("model", estimator)])

    if algorithm == "Random Forest":
        estimator = RandomForestClassifier(
            class_weight=class_weights,
            n_jobs=2,
            random_state=RANDOM_STATE,
            **params
        )
        return estimator

    if algorithm == "XGBoost":
        estimator = XGBClassifier(
            objective="multi:softprob",
            num_class=len(classes),
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=2,
            random_state=RANDOM_STATE,
            **params
        )
        return estimator

    raise ValueError(f"Unknown algorithm: {algorithm}")

def fit_one_model(algorithm, params, X_train, y_train,
                  undersampling=False, class_weighting=False, seed=RANDOM_STATE):
    y_train_array = np.asarray(y_train, dtype=int)
    weights_by_class = (
        original_fold_class_weights(y_train_array) if class_weighting else None
    )

    if undersampling:
        sampler = RandomUnderSampler(random_state=seed)
        X_fit, y_fit = sampler.fit_resample(X_train, y_train_array)
    else:
        X_fit, y_fit = X_train, y_train_array

    estimator = build_estimator(algorithm, params, class_weights=weights_by_class)

    fit_kwargs = {}
    if algorithm == "XGBoost" and class_weighting:
        # Apply weights calculated on the original fold, even if rows were undersampled.
        fit_kwargs["sample_weight"] = np.asarray(
            [weights_by_class[int(label)] for label in y_fit], dtype=float
        )

    estimator.fit(X_fit, y_fit, **(
        {"model__sample_weight": fit_kwargs["sample_weight"]}
        if algorithm == "XGBoost" and isinstance(estimator, Pipeline) and fit_kwargs
        else fit_kwargs
    ))
    return estimator

def score_predictions(y_true, y_pred):
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
    }

def get_roc_scores(fitted_model, X_eval, algorithm):
    # SVC has no predict_proba when probability=False; decision_function is suitable for ROC.
    if algorithm == "SVM (RBF)":
        return fitted_model.decision_function(X_eval)
    return fitted_model.predict_proba(X_eval)

## 11. Run the single integrated hyperparameter-tuning process

For each candidate, fold metrics are recorded individually and as means. Within each of the 16 combinations, the best candidate is selected by:
1. Mean validation Macro F1 (descending)
2. Mean validation balanced accuracy (descending)
3. Mean validation accuracy (descending)

The overall selected configuration is then the best of those 16 winners under the same ranking. No test-set information enters this process.

In [ ]:
candidate_records = []
fold_metric_records = []
candidate_counter = 0
total_candidates = len(ALGORITHMS) * len(IMBALANCE_SETTINGS) * N_ITER
search_start = time.time()

for algorithm in ALGORITHMS:
    sampled_params = list(ParameterSampler(
        PARAM_DISTRIBUTIONS[algorithm],
        n_iter=N_ITER,
        random_state=RANDOM_STATE
    ))

    for imbalance in IMBALANCE_SETTINGS:
        for candidate_number, params in enumerate(sampled_params, start=1):
            candidate_counter += 1
            fold_scores = []

            for fold_id, (train_idx, val_idx, train_yrs, val_yrs) in enumerate(folds, start=1):
                X_tr, y_tr = X_dev.iloc[train_idx], y_dev.iloc[train_idx]
                X_val, y_val = X_dev.iloc[val_idx], y_dev.iloc[val_idx]

                fitted = fit_one_model(
                    algorithm=algorithm,
                    params=params,
                    X_train=X_tr,
                    y_train=y_tr,
                    undersampling=imbalance["undersampling"],
                    class_weighting=imbalance["class_weighting"],
                    seed=RANDOM_STATE + fold_id
                )
                val_pred = fitted.predict(X_val)
                metrics = score_predictions(y_val, val_pred)

                fold_record = {
                    "algorithm": algorithm,
                    "undersampling": imbalance["undersampling"],
                    "class_weighting": imbalance["class_weighting"],
                    "candidate": candidate_number,
                    "fold": fold_id,
                    "training_period": f"{min(train_yrs)}-{max(train_yrs)}",
                    "validation_period": f"{min(val_yrs)}-{max(val_yrs)}",
                    **metrics
                }
                fold_metric_records.append(fold_record)
                fold_scores.append(metrics)

            candidate_records.append({
                "algorithm": algorithm,
                "undersampling": imbalance["undersampling"],
                "class_weighting": imbalance["class_weighting"],
                "candidate": candidate_number,
                "params": params,
                "mean_validation_macro_f1": np.mean([m["macro_f1"] for m in fold_scores]),
                "std_validation_macro_f1": np.std([m["macro_f1"] for m in fold_scores], ddof=1),
                "mean_validation_balanced_accuracy": np.mean([m["balanced_accuracy"] for m in fold_scores]),
                "mean_validation_accuracy": np.mean([m["accuracy"] for m in fold_scores]),
            })

            if candidate_counter % 8 == 0 or candidate_counter == total_candidates:
                elapsed = (time.time() - search_start) / 60
                print(f"Completed {candidate_counter}/{total_candidates} candidates "
                      f"({elapsed:.1f} min elapsed)")

candidate_results = pd.DataFrame(candidate_records)
fold_metrics_df = pd.DataFrame(fold_metric_records)

candidate_results = candidate_results.sort_values(
    ["mean_validation_macro_f1",
     "mean_validation_balanced_accuracy",
     "mean_validation_accuracy"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Tuning complete. Candidate rows:", len(candidate_results))
display(candidate_results.drop(columns="params").head(20))

## 12. Select the best hyperparameter configuration in each of the 16 combinations

This creates one consolidated comparison table. The best configuration in each combination is selected only from the shared chronological validation folds.

In [ ]:
combination_keys = ["algorithm", "undersampling", "class_weighting"]
best_per_combination = (
    candidate_results
    .sort_values(
        ["mean_validation_macro_f1",
         "mean_validation_balanced_accuracy",
         "mean_validation_accuracy"],
        ascending=[False, False, False]
    )
    .groupby(combination_keys, as_index=False, sort=False)
    .first()
)

best_per_combination = best_per_combination.sort_values(
    ["mean_validation_macro_f1",
     "mean_validation_balanced_accuracy",
     "mean_validation_accuracy"],
    ascending=[False, False, False]
).reset_index(drop=True)

best_per_combination.insert(0, "validation_rank", np.arange(1, len(best_per_combination) + 1))
assert len(best_per_combination) == 16, (
    f"Expected 16 best configurations, got {len(best_per_combination)}"
)

display(best_per_combination.drop(columns="params").round(4))

## 13. Train-versus-validation plots for all 16 selected configurations

For each selected configuration, the model is refitted on each fold's training data. Training metrics are calculated on the **original, non-undersampled training fold**, not on the resampled training subset. Validation metrics are calculated on the untouched validation fold.

The plots show accuracy and macro F1 so that a model that appears strong only on the majority class is easier to identify.

In [ ]:
train_validation_records = []

for _, row in best_per_combination.iterrows():
    algorithm = row["algorithm"]
    params = row["params"]
    under = bool(row["undersampling"])
    weight = bool(row["class_weighting"])

    for fold_id, (train_idx, val_idx, train_yrs, val_yrs) in enumerate(folds, start=1):
        X_tr, y_tr = X_dev.iloc[train_idx], y_dev.iloc[train_idx]
        X_val, y_val = X_dev.iloc[val_idx], y_dev.iloc[val_idx]

        fitted = fit_one_model(
            algorithm, params, X_tr, y_tr,
            undersampling=under,
            class_weighting=weight,
            seed=RANDOM_STATE + 1000 + fold_id
        )
        train_pred = fitted.predict(X_tr)
        val_pred = fitted.predict(X_val)
        train_m = score_predictions(y_tr, train_pred)
        val_m = score_predictions(y_val, val_pred)

        train_validation_records.append({
            "algorithm": algorithm,
            "undersampling": under,
            "class_weighting": weight,
            "fold": fold_id,
            "training_period": f"{min(train_yrs)}-{max(train_yrs)}",
            "validation_period": f"{min(val_yrs)}-{max(val_yrs)}",
            "train_accuracy": train_m["accuracy"],
            "validation_accuracy": val_m["accuracy"],
            "train_macro_f1": train_m["macro_f1"],
            "validation_macro_f1": val_m["macro_f1"],
        })

train_validation_df = pd.DataFrame(train_validation_records)
display(train_validation_df.head(10))

for _, row in best_per_combination.iterrows():
    mask = (
        (train_validation_df["algorithm"] == row["algorithm"]) &
        (train_validation_df["undersampling"] == row["undersampling"]) &
        (train_validation_df["class_weighting"] == row["class_weighting"])
    )
    sub = train_validation_df.loc[mask].sort_values("fold")
    label = (f'{row["algorithm"]} | undersampling={row["undersampling"]} | '
             f'class_weighting={row["class_weighting"]}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(sub["fold"], sub["train_accuracy"], marker="o", label="Train")
    axes[0].plot(sub["fold"], sub["validation_accuracy"], marker="o", label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_ylim(0, 1)
    axes[0].set_xlabel("Expanding-window fold")
    axes[0].legend()

    axes[1].plot(sub["fold"], sub["train_macro_f1"], marker="o", label="Train")
    axes[1].plot(sub["fold"], sub["validation_macro_f1"], marker="o", label="Validation")
    axes[1].set_title("Macro F1")
    axes[1].set_ylim(0, 1)
    axes[1].set_xlabel("Expanding-window fold")
    axes[1].legend()

    fig.suptitle(label, fontsize=10)
    plt.tight_layout()
    safe = "".join(ch if ch.isalnum() else "_" for ch in label.lower()).strip("_")
    fig.savefig(FIGURE_DIR / f"train_vs_validation_{safe}.png", dpi=160, bbox_inches="tight")
    plt.show()

## 14. Final fit and untouched 2020–2024 test evaluation

All 16 selected configurations are refitted using the complete development period (1991–2019) and then evaluated on the same untouched test set (2020–2024). Undersampling and class weighting are applied only during fitting.

The test metrics are reported for comparison; **they do not change the validation-based winner**.

In [ ]:
final_models = {}
test_predictions = {}
test_roc_scores = {}
test_metric_rows = []
test_class_labels = np.arange(len(classes))

for _, row in best_per_combination.iterrows():
    algorithm = row["algorithm"]
    under = bool(row["undersampling"])
    weight = bool(row["class_weighting"])
    params = row["params"]

    config_name = (
        f"{algorithm} | undersampling={under} | class_weighting={weight}"
    )

    final_model = fit_one_model(
        algorithm, params, X_dev, y_dev,
        undersampling=under,
        class_weighting=weight,
        seed=RANDOM_STATE + 9999
    )

    pred = final_model.predict(X_test)
    roc_scores = get_roc_scores(final_model, X_test, algorithm)

    final_models[config_name] = final_model
    test_predictions[config_name] = pred
    test_roc_scores[config_name] = roc_scores

    test_metric_rows.append({
        "configuration": config_name,
        "algorithm": algorithm,
        "undersampling": under,
        "class_weighting": weight,
        "validation_macro_f1": row["mean_validation_macro_f1"],
        "validation_balanced_accuracy": row["mean_validation_balanced_accuracy"],
        "validation_accuracy": row["mean_validation_accuracy"],
        "test_macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "test_accuracy": accuracy_score(y_test, pred),
        "test_macro_precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "test_macro_recall": recall_score(y_test, pred, average="macro", zero_division=0),
    })

final_test_metrics = pd.DataFrame(test_metric_rows)

# Winner is chosen strictly from validation performance, not test performance.
selected_row = best_per_combination.iloc[0]
selected_config = (
    f'{selected_row["algorithm"]} | undersampling={bool(selected_row["undersampling"])} | '
    f'class_weighting={bool(selected_row["class_weighting"])}'
)
selected_model = final_models[selected_config]

print("Selected configuration (validation only):")
print(selected_config)
print("Validation Macro F1:", round(float(selected_row["mean_validation_macro_f1"]), 4))
print("\nFinal test comparison (sorted by validation rank):")
display(final_test_metrics.merge(
    best_per_combination[combination_keys + ["validation_rank"]],
    on=combination_keys,
    how="left"
).sort_values("validation_rank").round(4))

## 15. Test-set confusion matrices and classification reports

Both raw and row-normalized confusion matrices are generated for each of the 16 configurations. The classification reports use the full known class list, so absent classes are explicitly represented with zero support.

In [ ]:
OUTPUT_DIR = Path("glacier_classification_outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

class_names = [int_to_class[i] for i in range(len(classes))]
classification_report_rows = []

for config_name, pred in test_predictions.items():
    print("\n" + "=" * 90)
    print(config_name)
    report = classification_report(
        y_test, pred,
        labels=test_class_labels,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    print(classification_report(
        y_test, pred,
        labels=test_class_labels,
        target_names=class_names,
        zero_division=0
    ))

    for class_name, values in report.items():
        if isinstance(values, dict):
            classification_report_rows.append({
                "configuration": config_name,
                "class": class_name,
                **values
            })

    cm = confusion_matrix(y_test, pred, labels=test_class_labels)
    cm_norm = confusion_matrix(
        y_test, pred, labels=test_class_labels, normalize="true"
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title("Raw counts")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")

    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title("Row-normalized")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")

    fig.suptitle(config_name, fontsize=10)
    plt.tight_layout()
    safe = "".join(ch if ch.isalnum() else "_" for ch in config_name.lower()).strip("_")
    fig.savefig(FIGURE_DIR / f"confusion_matrix_{safe}.png", dpi=160, bbox_inches="tight")
    plt.show()

classification_reports_df = pd.DataFrame(classification_report_rows)

## 16. Multiclass ROC curves (One-vs-Rest)

ROC scores come from predicted probabilities for Logistic Regression, Random Forest and XGBoost, and decision-function scores for RBF-SVM. A class-specific ROC curve is drawn only when the test set contains both positive and negative observations for that class.

The supplied 2020–2024 test distribution has very few or no observations from some classes in later years; therefore, some class ROC curves and aggregate multiclass AUC values may be undefined. The notebook reports this rather than inventing a curve or score.

In [ ]:
roc_auc_rows = []

for config_name, scores in test_roc_scores.items():
    scores = np.asarray(scores)
    fig, ax = plt.subplots(figsize=(8, 6))
    plotted = 0

    for class_idx, class_name in enumerate(class_names):
        y_binary = (y_test.to_numpy() == class_idx).astype(int)
        if len(np.unique(y_binary)) < 2:
            roc_auc_rows.append({
                "configuration": config_name,
                "class": class_name,
                "roc_auc": np.nan,
                "status": "Undefined: test class has only one outcome"
            })
            continue

        # For multiclass SVC decision_function, output columns correspond to estimator classes.
        class_scores = scores[:, class_idx]
        fpr, tpr, _ = roc_curve(y_binary, class_scores)
        class_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{class_name} (AUC={class_auc:.3f})")
        roc_auc_rows.append({
            "configuration": config_name,
            "class": class_name,
            "roc_auc": class_auc,
            "status": "Defined"
        })
        plotted += 1

    if plotted:
        ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"Test ROC — {config_name}")
        ax.legend(loc="lower right")
        plt.tight_layout()
        safe = "".join(ch if ch.isalnum() else "_" for ch in config_name.lower()).strip("_")
        fig.savefig(FIGURE_DIR / f"roc_{safe}.png", dpi=160, bbox_inches="tight")
        plt.show()
    else:
        plt.close(fig)
        print("No defined class ROC curves for:", config_name)

roc_auc_df = pd.DataFrame(roc_auc_rows)

## 17. Overall comparisons and selected-model diagnostics

The primary ranking remains validation Macro F1. Test Macro F1, test accuracy and test balanced accuracy are secondary descriptive results and must not be used to re-select the model.

In [ ]:
comparison = final_test_metrics.merge(
    best_per_combination[combination_keys + ["validation_rank"]],
    on=combination_keys,
    how="left"
).sort_values("validation_rank")

display(comparison[[
    "validation_rank", "algorithm", "undersampling", "class_weighting",
    "validation_macro_f1", "validation_balanced_accuracy", "validation_accuracy",
    "test_macro_f1", "test_balanced_accuracy", "test_accuracy"
]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_df = comparison.copy()
plot_df["short_name"] = [
    f"{r.algorithm}\nU={int(r.undersampling)}, W={int(r.class_weighting)}"
    for r in plot_df.itertuples()
]
axes[0].bar(plot_df["short_name"], plot_df["test_macro_f1"])
axes[0].set_title("Test Macro F1 — all 16 selected configurations")
axes[0].set_ylabel("Macro F1")
axes[0].tick_params(axis="x", rotation=90)

axes[1].bar(plot_df["short_name"], plot_df["test_balanced_accuracy"], label="Balanced accuracy")
axes[1].plot(np.arange(len(plot_df)), plot_df["test_accuracy"], marker="o",
             linestyle="--", label="Accuracy")
axes[1].set_title("Test accuracy comparison")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=90)
axes[1].legend()
plt.tight_layout()
fig.savefig(FIGURE_DIR / "test_performance_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

print("Selected model test metrics:")
display(final_test_metrics.loc[
    final_test_metrics["configuration"] == selected_config
].round(4))

## 18. Export CSV reports, plots, metadata and Joblib model

The selected model is the validation-ranked winner. The Joblib artifact includes the fitted estimator/pipeline, feature order, class mappings, selected settings and validation scores. Keep the metadata JSON with the model.

In [ ]:
# Consolidated CSV reports
candidate_export = candidate_results.copy()
candidate_export["params"] = candidate_export["params"].apply(
    lambda value: json.dumps(value, default=str, sort_keys=True)
)
best_export = best_per_combination.copy()
best_export["params"] = best_export["params"].apply(
    lambda value: json.dumps(value, default=str, sort_keys=True)
)

candidate_export.to_csv(OUTPUT_DIR / "all_candidate_validation_results.csv", index=False)
fold_metrics_df.to_csv(OUTPUT_DIR / "all_candidate_fold_metrics.csv", index=False)
best_export.to_csv(OUTPUT_DIR / "best_of_16_configurations_validation.csv", index=False)
train_validation_df.to_csv(OUTPUT_DIR / "train_vs_validation_metrics.csv", index=False)
final_test_metrics.to_csv(OUTPUT_DIR / "final_test_metrics_all_16.csv", index=False)
classification_reports_df.to_csv(OUTPUT_DIR / "test_classification_reports.csv", index=False)
roc_auc_df.to_csv(OUTPUT_DIR / "test_roc_auc_by_class.csv", index=False)
fold_summary.to_csv(OUTPUT_DIR / "expanding_window_fold_schedule.csv", index=False)

metadata = {
    "selected_configuration": selected_config,
    "selection_metric": "mean expanding-window validation macro F1",
    "selected_algorithm": selected_row["algorithm"],
    "undersampling": bool(selected_row["undersampling"]),
    "class_weighting": bool(selected_row["class_weighting"]),
    "selected_hyperparameters": selected_row["params"],
    "validation_macro_f1": float(selected_row["mean_validation_macro_f1"]),
    "validation_balanced_accuracy": float(selected_row["mean_validation_balanced_accuracy"]),
    "validation_accuracy": float(selected_row["mean_validation_accuracy"]),
    "feature_columns": feature_cols,
    "excluded_columns": LEAKAGE_COLUMNS,
    "target_column": TARGET,
    "class_to_integer": class_to_int,
    "integer_to_class": {str(k): v for k, v in int_to_class.items()},
    "study_scope": "Entire HKH dataset; all supplied grid locations retained",
    "unique_grid_locations": int(n_locations),
    "development_years": [int(min(dev_years)), int(max(dev_years))],
    "test_years": [int(min(test_years)), int(max(test_years))],
    "random_state": RANDOM_STATE,
    "n_iter_per_algorithm_imbalance_combination": N_ITER,
    "number_of_combinations": 16,
    "validation_folds": fold_summary.to_dict(orient="records"),
    "target_thresholds_mwe": {
        "mass_gain": "> 0.1",
        "mass_loss": "< -0.1",
        "near_balance": "between -0.1 and +0.1 inclusive"
    },
    "note": (
        "Test metrics are descriptive and were not used for model selection. "
        "Undersampling was applied only to training data. Class weights, when enabled, "
        "were computed from the original pre-undersampling training-fold distribution."
    )
}

model_bundle = {
    "model": selected_model,
    "feature_columns": feature_cols,
    "class_to_integer": class_to_int,
    "integer_to_class": int_to_class,
    "metadata": metadata
}

model_path = OUTPUT_DIR / "selected_hkh_glacier_mass_change_classifier.joblib"
metadata_path = OUTPUT_DIR / "selected_model_metadata.json"

joblib.dump(model_bundle, model_path)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved outputs to:", OUTPUT_DIR.resolve())
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(OUTPUT_DIR)} | {path.stat().st_size / 1024:.1f} KB")

## 19. Reload and use the selected model

The saved artifact is a dictionary containing the fitted estimator and its feature/class metadata. The example below reloads it and predicts labels for a few held-out rows. These predictions do not alter model selection.

In [ ]:
loaded_bundle = joblib.load(OUTPUT_DIR / "selected_hkh_glacier_mass_change_classifier.joblib")
loaded_model = loaded_bundle["model"]
loaded_features = loaded_bundle["feature_columns"]
loaded_int_to_class = {int(k): v for k, v in loaded_bundle["integer_to_class"].items()}

example_X = X_test[loaded_features].head(5).copy()
example_codes = loaded_model.predict(example_X)
example_labels = [loaded_int_to_class[int(code)] for code in example_codes]

example_output = example_X.copy()
example_output["predicted_mass_change_class"] = example_labels
display(example_output)

## 20. Research limitations and interpretation checklist

1. **Spatial dependence:** grid cells may be spatially autocorrelated. The chronological split tests future years at represented locations, not spatial generalization to unseen locations. Consider spatial-block validation as a separate experiment.
2. **Changing class distribution:** inspect the strong temporal changes in class frequencies, especially the near-absence of gain/near-balance in some test years. This may reflect physical conditions or target/data construction; investigate rather than modifying the test labels.
3. **Target thresholds:** justify and cite the ±0.1 m w.e. classification thresholds. Results depend on these choices.
4. **Feature availability:** full hydrological-year climate aggregates are retrospective predictors unless available at the intended forecast origin. State the prediction time clearly.
5. **Undersampling randomness:** this notebook uses one fixed random seed for reproducibility. For a robust publication, repeat the entire tuning procedure with multiple seeds or assess seed sensitivity.
6. **Combined imbalance strategy:** undersampling plus weights based on the original fold may strongly amplify minority classes. The 16-way comparison makes this measurable; it is not an assumption that combined handling is best.
7. **ROC limitations:** a class absent from the final test period has undefined class-specific ROC-AUC. Do not interpret undefined values as zero or fabricate ROC curves.
8. **Computational cost:** default 192 candidate configurations × the number of chronological folds, followed by 16 final fits and 16 diagnostic refits. Reduce `N_ITER` for a smoke test, then restore it for the final run.
9. **No test-driven selection:** the winning model is selected only using validation Macro F1, with balanced accuracy and accuracy as tie-breakers. Report test results once as final evaluation.